# MedGuardAI — Fine-tune Gemma-3-4b with QLoRA on a free Colab/Kaggle T4

**Inputs:** `sft_pairs.jsonl` produced by `backend/training/build_qa_dataset.py` (synthetic Q&A grounded in the 9,879-drug FDA label corpus).

**Outputs:**
- LoRA adapter at `outputs/medguard-sft/` (~80 MB)
- Merged GGUF Q4_K_M at `outputs/medguard-gguf/` (~2.5 GB) — drop-in for LM Studio/Ollama

**Hardware:** Free Colab T4 (16 GB VRAM) is plenty. Kaggle's T4 also works.

**Runtime:** ~30–60 min for 1 epoch on ~3-5k pairs.

## 1. Install dependencies

Unsloth bundles bitsandbytes/peft/trl with patches that make Gemma-3 train ~2x faster and use ~50% less VRAM.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.12.0,<0.20.0" peft accelerate bitsandbytes
!pip install datasets sentencepiece protobuf

## 2. Upload the SFT dataset

Either upload `sft_pairs.jsonl` directly (small datasets) or mount Google Drive and read it from there.

**Option A (simplest):** click the "Files" panel on the left → upload `sft_pairs.jsonl` to `/content/`.
**Option B (persistent):** mount Drive and point `DATASET_PATH` at the file.

In [ ]:
DATASET_PATH = "/content/sft_pairs.jsonl"  # change if mounted from Drive

import os
assert os.path.exists(DATASET_PATH), f"upload sft_pairs.jsonl to {DATASET_PATH} first"

import json
with open(DATASET_PATH) as f:
    n = sum(1 for _ in f)
print(f"dataset has {n} pairs")

## 3. Load Gemma-3-4b in 4-bit via Unsloth

We use the *instruct* variant (`gemma-3-4b-it`) so the model already understands chat formatting; fine-tuning teaches it the FDA-grounding behavior on top.

In [ ]:
from unsloth import FastModel
import torch

MAX_SEQ_LENGTH = 2048  # FDA sections can be long

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
)
print("VRAM after loading:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

## 4. Apply LoRA adapters

We adapt the attention + MLP projections, which is the standard QLoRA recipe. `r=16` balances capacity and adapter size.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    random_state=42,
)

## 5. Format the dataset for chat training

Each sample becomes a 3-turn conversation: a clinical system prompt, the user's question (drug-prefixed), and the grounded answer. This matches how the agent will call the model in production.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

SYSTEM_PROMPT = (
    "You are MedGuardAI, a safety-first clinical assistant. Answer based on the "
    "FDA documentation context you are given. If the answer is not supported by "
    "the documentation, say so. Never hallucinate dosages or interactions."
)

raw = load_dataset("json", data_files=DATASET_PATH, split="train")
print("raw fields:", raw.column_names)

def to_chat(example):
    user_msg = f"Drug: {example['drug']}\nQuestion: {example['question']}"
    convo = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": user_msg}]},
        {"role": "assistant", "content": [{"type": "text", "text": example["answer"]}]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
    }

ds = raw.map(to_chat, remove_columns=raw.column_names)
print("\nsample formatted prompt:")
print(ds[0]["text"][:1000])

## 6. Train

1 epoch is usually enough for SFT on a few thousand pairs. Bump `num_train_epochs` to 2 if you want more memorization (at the cost of overfitting risk).

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs/checkpoints",
        save_strategy="no",
        report_to="none",
    ),
)

stats = trainer.train()
print(stats)

## 7. Save the LoRA adapter

`outputs/medguard-sft/` is small (~80 MB) and is what you want to keep for further DPO training in Phase 3.

In [ ]:
ADAPTER_DIR = "outputs/medguard-sft"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
!du -sh {ADAPTER_DIR}

## 8. Quick inference test

Before exporting, sanity-check that the fine-tuned model answers a held-out question reasonably.

In [ ]:
test_question = "For naproxen: What is the recommended starting dose for acute gout?"
convo = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user", "content": [{"type": "text", "text": test_question}]},
]
inputs = tokenizer.apply_chat_template(
    convo, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=200, do_sample=False, temperature=0.0)
print(tokenizer.batch_decode(out)[0])

## 9. Merge to fp16 and convert to GGUF Q4_K_M

GGUF is the format LM Studio / Ollama / llama.cpp consume directly. Q4_K_M is a good speed/quality balance — ~2.5 GB on disk, runs at ~30 tok/s on a modern CPU.

**Note:** GGUF conversion downloads `llama.cpp` and compiles it. First run takes ~5 min.

In [ ]:
GGUF_DIR = "outputs/medguard-gguf"

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)
!ls -la {GGUF_DIR}
!du -sh {GGUF_DIR}

## 10. Download or push to Hugging Face Hub

Two options for getting the artifacts off Colab:

### 10a. Direct download (zip + browser download)

In [ ]:
!cd outputs && zip -r medguard-sft.zip medguard-sft
!cd outputs && zip -r medguard-gguf.zip medguard-gguf

from google.colab import files
files.download("outputs/medguard-sft.zip")
files.download("outputs/medguard-gguf.zip")

### 10b. Push to Hugging Face Hub (private repo, easy team sharing)

Generate a write token at https://huggingface.co/settings/tokens. Set the username and repo name below.

In [ ]:
from huggingface_hub import login, create_repo, upload_folder

HF_TOKEN = "hf_..."  # paste your token (or use getpass)
HF_USER = "your-username"
REPO = f"{HF_USER}/medguardai-gemma-3-4b"

login(token=HF_TOKEN)
create_repo(REPO, private=True, exist_ok=True)

# Push the LoRA adapter
upload_folder(
    repo_id=REPO,
    folder_path=ADAPTER_DIR,
    path_in_repo="sft-adapter",
)
# Push the GGUF
upload_folder(
    repo_id=REPO,
    folder_path=GGUF_DIR,
    path_in_repo="gguf",
)
print(f"\npushed to https://huggingface.co/{REPO}")

## 11. Use in MedGuardAI

After downloading the GGUF locally:

1. Drop `medguard-gemma-3-4b-q4_k_m.gguf` into LM Studio's models folder (Settings → "My Models" → the directory shown there).
2. In LM Studio, click "Load Model" → select the new file.
3. Update `.env` in the MedGuardAI repo:
   ```
   LOCAL_LLM_MODEL=medguard-gemma-3-4b
   ```
4. Restart the backend. The agent will now use your fine-tuned model with no other code changes.